# 02 — Scale Disparity: StandardScaler vs RobustScaler

**Goal:** prove Euclidean bias (Standpipe Pressure ~thousands psi vs Torque ~tens) and compare scalers for IEEE §II.
Uses 20k sample + K-Means(k=4) demo; full run in notebook 04.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

SENSOR_8 = ['Rate Of Penetration (ft_per_hr)','Weight on Bit (klbs)','Rotary RPM (RPM)','Standpipe Pressure (psi)','Rotary Torque (kft_lb)','Hook Load (klbs)','Total Pump Output (gal_per_min)','Differential Pressure (psi)']
CSV_NAME='56-32 1sec data 27029986.csv'

def resolve_input(csv=CSV_NAME):
    root = Path('/kaggle/input')
    if root.exists():
        hits = list(root.rglob(csv))
        if hits:
            return hits[0]
        for p in root.rglob('*.csv'):
            if '56-32' in p.name or '1sec' in p.name or '27029986' in p.name:
                return p
        csvs = [p for p in root.rglob('*.csv') if p.is_file()]
        if csvs:
            return max(csvs, key=lambda p: p.stat().st_size)
        pqs = [p for p in root.rglob('*.parquet') if p.is_file()]
        if pqs:
            return max(pqs, key=lambda p: p.stat().st_size)
    for c in [Path('dataset')/csv, Path('../dataset')/csv, Path.cwd()/'dataset'/csv]:
        if c.exists(): return c
    listing = [str(p) for p in sorted(root.rglob('*'))][:30] if root.exists() else ['</kaggle/input not found>']
    raise FileNotFoundError('Dataset CSV not found. /kaggle/input contains: ' + str(listing) + ' | Fix: right panel Add Input -> add your private dataset -> re-run.')
if Path('/kaggle/working').exists(): OUT = Path('/kaggle/working')  # live session: visible in file browser, downloadable, saved on Save Version
elif Path('/kaggle/output').exists(): OUT = Path('/kaggle/output')
else: OUT = Path('../figures')
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
df=pd.read_csv(resolve_input(), usecols=SENSOR_8, nrows=40000, low_memory=True)
for c in SENSOR_8: df[c]=pd.to_numeric(df[c], errors='coerce').astype('float32')
df=df.replace(-999.25, np.nan).dropna().reset_index(drop=True)
demo=df.sample(20000, random_state=42).to_numpy()
print('std per sensor (raw):')
print(pd.DataFrame({'sensor':SENSOR_8,'std':demo.std(axis=0)}).sort_values('std', ascending=False).to_string(index=False))

In [ ]:
res={}
for name, scaler in [('unscaled',None),('standard',StandardScaler()),('robust',RobustScaler())]:
    X = demo if scaler is None else scaler.fit_transform(demo)
    lab = KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(X)
    res[name]={'silhouette':float(silhouette_score(X,lab)), 'dbi':float(davies_bouldin_score(X,lab))}
    print(name, res[name])
pd.DataFrame(res).T.round(4).to_csv(OUT/'table2_scaler_compare.csv')
print('saved table2_scaler_compare.csv — higher silhouette better, lower DBI better')

In [ ]:
xs, ss, rs = demo, StandardScaler().fit_transform(demo), RobustScaler().fit_transform(demo)
fig, ax = plt.subplots(1,3, figsize=(12,3), sharey=True)
for a, X, t in zip(ax,[xs,ss,rs],['Raw','StandardScaler','RobustScaler']):
    a.hist(X.ravel(), bins=80); a.set_title(t); a.set_yscale('log')
plt.suptitle('Value Distribution: Raw vs Scaled (log-y)')
plt.tight_layout(); plt.savefig(OUT/'fig3_scaling_hist.png', dpi=150); plt.show()

#### Cell: Export & verify — collect everything for download
**What:** lists what was actually written to `OUT`, copies it into `/kaggle/working` (the live, downloadable folder), and packs it into one `kaggle_outputs.zip`.
**Why:** if this cell reports `<EMPTY>`, the cells above never ran (error or skipped) — outputs cannot exist. The zip gives you one-click download instead of per-file clicks.
**Read this:** the printed file list (expect `table*.csv` + `fig*.png`) and the zip size; then download via the file browser.


In [ ]:
from pathlib import Path
import shutil, zipfile
OUT.mkdir(parents=True, exist_ok=True)
live = Path('/kaggle/working') if Path('/kaggle/working').exists() else OUT
moved = []
for f in sorted(OUT.glob('*')):
    if not f.is_file() or f.suffix == '.zip':
        continue
    t = live / f.name
    if f.resolve() != t.resolve():
        shutil.copy2(f, t)
    moved.append(f.name)
print('OUT =', OUT)
print('files:', moved if moved else '<EMPTY — cells above did not run or errored, re-run from the top>')
zp = live / 'kaggle_outputs.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in moved:
        z.write(live / name, name)
print(f'zip: {zp} ({zp.stat().st_size/1024:.1f} KB)')
print('Download: file browser ->', live, '-> kaggle_outputs.zip. For the paper link: Save Version -> Output tab.')

try:
    from IPython.display import FileLink, display as _display
    _display(FileLink(str(zp)))
    print('Click the link above to download the zip directly (no file browser needed).')
except Exception as _e:
    print('direct-link unavailable:', _e)
